In [32]:
import torch
from torchvision import datasets, transforms
import pytorch_lightning as pl
from torch import nn
torch.set_float32_matmul_precision('high')

In [34]:
def get_dataloader(dataset_name, batch_size, n_splits):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name.lower() == "mnist":
        dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    elif dataset_name.lower() == "cifar10":
        dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    else:
        raise ValueError(f"Dataset {dataset_name} not supported.")

    # Split the dataset into n_splits
    dataset_size = len(dataset)
    indices = torch.randperm(dataset_size).tolist()

    split_size = dataset_size // n_splits
    splits = [indices[i * split_size:(i + 1) * split_size] for i in range(n_splits)]
    train_loaders = []
    for i in range(len(splits)):
        splits[i] = torch.utils.data.Subset(dataset, splits[i])
        train_loader = torch.utils.data.DataLoader(splits[0], batch_size=batch_size, shuffle=True, num_workers=3, persistent_workers=True)
        train_loaders.append(train_loader)

    return train_loaders


In [35]:
train_loaders = get_dataloader("cifar10", 128, 5)

base_model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(32 * 16 * 16, 512),
    nn.ReLU(),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

Files already downloaded and verified


In [37]:
from Componenets import FederatedModelWrapper

model = FederatedModelWrapper(base_model)
worker_models_list = [FederatedModelWrapper(base_model) for i in range(len(train_loaders))]

# server send model to nodes
server_model_to_send = model.state_dict()

In [47]:
import importlib
import Componenets
importlib.reload(Componenets)
from Componenets import grad_quantizer, grad_encoder, grad_decoder, grad_de_quantizer

def sim_worker(model, train_loader):
    model.load_state_dict(server_model_to_send)
    trainer = pl.Trainer(max_epochs=5, enable_progress_bar=False)
    trainer.fit(model, train_loader)
    
    quant = grad_quantizer(model.latest_parameters)
    encode = grad_encoder(quant)
    return quant, encode

def sim_server(quant_list, encode_list):
    decoded = grad_decoder(worker_encode)
    de_quant = grad_de_quantizer(decoded, temp)
    return de_quant

In [40]:
signals_data = {'raw':[], 'quant': [], 'encode': []}

# worker side
for i in range(len(train_loaders)):
    worker_quant, worker_encode = sim_worker(worker_models_list[i], train_loaders[i])
    signals_data['quant'].append(worker_quant)
    signals_data['encode'].append(worker_encode)
    signals_data['raw'].append(worker_models_list[i].latest_parameters)

# server side
s_decoded = grad_decoder(signals_data['encode'])
temp=[[a,b.shape] for a,b in worker_models_list[0].latest_parameters.copy()]
s_de_quant = grad_de_quantizer(s_decoded, temp)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params
---------------------------------------------
0 | model   | Sequential       | 4.3 M 
1 | loss_fn | CrossEntropyLoss | 0     
---------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.081    Total estimated model params size (MB)
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params
---------------------------------------------
0 | model   | Sequential       | 4.3 M 
1 | loss_fn | CrossEntropyLoss | 0     
---------------------------------------------
4.3 M    

NameError: name 'server_agg' is not defined

In [49]:
# aggregate
worker_count=len(train_loaders)
final_grad = [[s_de_quant[0][i][0], s_de_quant[0][i][1]/worker_count] for i in range(len(s_de_quant[0]))]
for i in range(1, len(s_de_quant)):
    for j in range(len(final_grad)):
        final_grad[j][1] += s_de_quant[i][j][1]/worker_count

In [50]:
# update server model via optimizer
optimizer = model.configure_optimizers()
optimizer.zero_grad()
for i in range(len(final_grad)):
    name, grad = final_grad[i]
    param = dict(model.named_parameters())['model.'+name]
    param.grad = torch.tensor(grad).to(param.device)
optimizer.step()